In [1]:
!pip install openai chromadb pypdf tiktoken python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 70.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 51.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.1 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    F

In [6]:
import os
from openai import OpenAI
import chromadb
from pypdf import PdfReader

# -------------------------------
# Setup
# -------------------------------
import os

os.environ["OPENAI_API_KEY"] = "sk-proj"
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# -------------------------------
# Step 1: Load PDF
# -------------------------------
def load_pdf(file_path):
    reader = PdfReader(file_path)
    text = ""
    for page in reader.pages:
        text += page.extract_text() + "\n"
    return text

# -------------------------------
# Step 2: Chunk Text
# -------------------------------
def chunk_text(text, chunk_size=500, overlap=50):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

# -------------------------------
# Step 3: Create Embeddings
# -------------------------------
def get_embedding(text):
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return response.data[0].embedding

# -------------------------------
# Step 4: Store in ChromaDB
# -------------------------------
def create_vector_db(chunks):
    chroma_client = chromadb.Client()
    collection = chroma_client.create_collection(name="rag_demo")

    for i, chunk in enumerate(chunks):
        embedding = get_embedding(chunk)
        collection.add(
            ids=[str(i)],
            embeddings=[embedding],
            documents=[chunk]
        )

    return collection

# -------------------------------
# Step 5: Retrieve Relevant Docs
# -------------------------------
def retrieve(query, collection, top_k=3):
    query_embedding = get_embedding(query)

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )

    return results["documents"][0]
        # Print retrieved chunks
    print("\nRetrieved Chunks:\n" + "-"*50)
    for i, doc in enumerate(docs):
        print(f"\nChunk {i+1}:\n{doc[:300]}...")  # show first 300 chars

    return docs

# -------------------------------
# Step 6: Generate Answer
# -------------------------------
def generate_answer(query, context_docs):
    context = "\n\n".join(context_docs)

    prompt = f"""
You are a helpful assistant.
Answer the question based ONLY on the context below.

Context:
{context}

Question:
{query}
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0.2
    )

    return response.choices[0].message.content



In [7]:
# -------------------------------
# MAIN PIPELINE
# -------------------------------
if __name__ == "__main__":
    pdf_path = "sample.pdf"

    print("📄 Loading PDF...")
    text = load_pdf(pdf_path)

    print("✂️ Chunking...")
    chunks = chunk_text(text)

    print("🧠 Creating Vector DB...")
    collection = create_vector_db(chunks)

    print("✅ Ready for questions!")



📄 Loading PDF...
✂️ Chunking...
🧠 Creating Vector DB...
✅ Ready for questions!

❓ Ask a question (or type 'exit'): leave policy

💡 Answer:
The leave policy outlined in the HR Policy Manual 2023 includes the following key points:

1. **Earned Leave Accrual**: Employees earn leave at the rate of two and a half days per completed calendar month up to the date of retirement or resignation.

2. **Leave Credit upon Dismissal**: If an employee is removed or dismissed, they will receive credit for earned leave at the same rate (two and a half days per completed calendar month) up to the end of the calendar month preceding their dismissal.

3. **Leave Credit upon Death**: In the event of an employee's death while in service, credit for earned leave will also be provided.

4. **Reduction of Earned Leave**: Earned Leave (EL) will be reduced by 1/10th of the Earned Leave taken and/or the period of non-availability during the previous half year, with a maximum reduction of 15 days.

5. **Applicatio

In [8]:
if __name__ == "__main__":
    while True:
          query = input("\n❓ Ask a question (or type 'exit'): ")
          if query.lower() == "exit":
              break

          docs = retrieve(query, collection)
          answer = generate_answer(query, docs)

          print("\n💡 Answer:")
          print(answer)


❓ Ask a question (or type 'exit'): Resignation

💡 Answer:
Employees of the Institute may resign from the Institute as per the provisions contained in their appointment letter. While the Institute is interested in settling all the outstanding dues as early as possible, no outstanding dues will be settled unless a properly endorsed clearance form is completed.

❓ Ask a question (or type 'exit'): exit


In [ ]:
# ================================
# SIMPLE RAG IN ONE CELL (COLAB)
# ================================

# Install dependencies
!pip install -q openai chromadb pypdf tiktoken

# -------------------------------
# Setup
# -------------------------------
import os
from openai import OpenAI
import chromadb
from pypdf import PdfReader
from google.colab import files

# Set your OpenAI API Key
os.environ["OPENAI_API_KEY"] = "your-api-key-here"
client = OpenAI()

# -------------------------------
# Upload PDF
# -------------------------------
print("Upload your PDF")
uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]

# -------------------------------
# Load PDF
# -------------------------------
def load_pdf(file_path):
    reader = PdfReader(file_path)
    text = ""
    for page in reader.pages:
        content = page.extract_text()
        if content:
            text += content + "\n"
    return text

# -------------------------------
# Chunk Text
# -------------------------------
def chunk_text(text, chunk_size=500, overlap=50):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

# -------------------------------
# Embedding
# -------------------------------
def get_embedding(text):
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return response.data[0].embedding

# -------------------------------
# Create Vector DB
# -------------------------------
chroma_client = chromadb.Client()
collection = chroma_client.create_collection(name="rag_demo")

# -------------------------------
# Build Index
# -------------------------------
print("Loading PDF...")
text = load_pdf(pdf_path)

print("Chunking...")
chunks = chunk_text(text)

print("Creating embeddings and storing...")
for i, chunk in enumerate(chunks):
    emb = get_embedding(chunk)
    collection.add(
        ids=[str(i)],
        embeddings=[emb],
        documents=[chunk]
    )

print("RAG Ready")

# -------------------------------
# Retriever
# -------------------------------
def retrieve(query, top_k=3):
    query_emb = get_embedding(query)
    results = collection.query(
        query_embeddings=[query_emb],
        n_results=top_k
    )
    return results["documents"][0]

# -------------------------------
# Generator (LLM)
# -------------------------------
def generate_answer(query, docs):
    context = "\n\n".join(docs)

    prompt = f"""
Answer ONLY from the given context.

Context:
{context}

Question:
{query}
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2
    )

    return response.choices[0].message.content

# -------------------------------
# Ask Questions Loop
# -------------------------------
while True:
    q = input("Ask (type 'exit' to stop): ")
    if q.lower() == "exit":
        break

    docs = retrieve(q)
    ans = generate_answer(q, docs)

    print("Answer:")
    print(ans)

In [ ]:
# ================================
# SIMPLE RAG WITH STREAMING (COLAB)
# ================================

# Install dependencies
!pip install -q openai chromadb pypdf tiktoken

# -------------------------------
# Setup
# -------------------------------
import os
from openai import OpenAI
import chromadb
from pypdf import PdfReader
from google.colab import files

# Set your OpenAI API Key
os.environ["OPENAI_API_KEY"] = "your-api-key-here"
client = OpenAI()

# -------------------------------
# Upload PDF
# -------------------------------
print("Upload your PDF")
uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]

# -------------------------------
# Load PDF
# -------------------------------
def load_pdf(file_path):
    reader = PdfReader(file_path)
    text = ""
    for page in reader.pages:
        content = page.extract_text()
        if content:
            text += content + "\n"
    return text

# -------------------------------
# Chunk Text
# -------------------------------
def chunk_text(text, chunk_size=500, overlap=50):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

# -------------------------------
# Embedding
# -------------------------------
def get_embedding(text):
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return response.data[0].embedding

# -------------------------------
# Create Vector DB
# -------------------------------
chroma_client = chromadb.Client()
collection = chroma_client.create_collection(name="rag_demo")

# -------------------------------
# Build Index
# -------------------------------
print("Loading PDF...")
text = load_pdf(pdf_path)

print("Chunking...")
chunks = chunk_text(text)

print("Creating embeddings and storing...")
for i, chunk in enumerate(chunks):
    emb = get_embedding(chunk)
    collection.add(
        ids=[str(i)],
        embeddings=[emb],
        documents=[chunk]
    )

print("RAG Ready")

# -------------------------------
# Retriever
# -------------------------------
def retrieve(query, top_k=3):
    query_emb = get_embedding(query)
    results = collection.query(
        query_embeddings=[query_emb],
        n_results=top_k
    )
    return results["documents"][0]

# -------------------------------
# Streaming Generator
# -------------------------------
def stream_answer(query, docs):
    context = "\n\n".join(docs)

    prompt = f"""
Answer ONLY from the given context.

Context:
{context}

Question:
{query}
"""

    stream = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
        stream=True
    )

    full_text = ""
    for chunk in stream:
        if chunk.choices[0].delta.content:
            token = chunk.choices[0].delta.content
            print(token, end="", flush=True)
            full_text += token

    print()  # newline after streaming
    return full_text

# -------------------------------
# Ask Questions Loop
# -------------------------------
while True:
    q = input("Ask (type 'exit' to stop): ")
    if q.lower() == "exit":
        break

    docs = retrieve(q)
    print("Answer:")
    stream_answer(q, docs)